# NB13: the unified company dataset, July bulk joined to the July Gazette features

The engineered company bulk and the Gazette distress features have lived in separate files, keyed on
`CompanyNumber`, so anything wanting both had to do the join itself. This notebook does it once and
writes one file, so a company lookup becomes a row lookup.

**What goes in**

- `data/raw/dashboard_bulk_2026-07.parquet`, 1,531,094 companies by 69 columns, one row per company,
  built in DuckDB. This is the spine: segmentation, charges and debt, filing compliance, change
  events, the LBG and competitor block, and the four scores.
- `data/processed/nb10_gazette_company_features_thru_2026-07.csv`, 109,333 companies by 56 columns,
  aggregated from 291,045 Gazette notices published between 2023-06-30 and 2026-07-31.

**What does not go in, and why**

- `nb10_gazette_company_features.csv` (the June build) is a strict subset of the July file.
- `nb10_gazette_notices_thru_2026-07.csv` is at notice grain, up to 8 rows per company. It is the
  source these features were aggregated from, so joining it would multiply rows. It stays separate.
- `nb10_active_gazette_watchlist_thru_2026-07.csv` is a subset of the features file. Its only extra
  column, `lifecycle`, comes from the widened universe CSV built on the 1 August snapshot, not from
  the Gazette, so it is not pulled into a July spine.
- `company_master_gazette_thru_2026-07.csv` is the earlier NB12 merge against the raw universe. All
  54 of its `gaz_` columns are already in the features file.

**The one thing to understand about the dates**

`base_month` is a month label, not an instant, in the same style as `contracts_asof_month`. The bulk
describes July 2026 and the Gazette runs to 2026-07-31, so the notices sit inside the same month
rather than after it. Nothing needs censoring. A July notice confirms distress in July, it does not
predict it, so these columns are contemporaneous with the spine rather than a future outcome. Using
July features to predict an August or September outcome carries no leakage.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

RAW = Path(r"C:\Users\visha\Lloyds_Github\data\raw")
PROCESSED = Path(r"C:\Users\visha\Lloyds_Github\data\processed")

SPINE = RAW / "dashboard_bulk_2026-07.parquet"
GAZ = PROCESSED / "nb10_gazette_company_features_thru_2026-07.csv"
OUT = PROCESSED / "dashboard_bulk_gazette_2026-07.parquet"
UNMATCHED_OUT = PROCESSED / "gaz_unmatched_2026-07.csv"

# The two clocks. They describe the same month but are not the same instant, so both
# are stamped into the output rather than left as notebook trivia.
BASE_MONTH = pd.Timestamp("2026-07-01")        # the bulk's month label
GAZ_ASOF = pd.Timestamp("2026-07-31")          # what every gaz_days_since_* is measured from
JULY_START, JULY_END = pd.Timestamp("2026-07-01"), pd.Timestamp("2026-07-31")

# Baselines measured during inspection. The asserts below are meant to fail loudly if an
# upstream file is rebuilt, rather than let a changed universe pass unnoticed.
EXPECT_SPINE_ROWS = 1_531_094
EXPECT_GAZ_ROWS = 109_333
EXPECT_MATCHED = 19_525

print("spine  :", SPINE)
print("gazette:", GAZ)
print("out    :", OUT)

spine  : C:\Users\visha\Lloyds_Github\data\raw\dashboard_bulk_2026-07.parquet
gazette: C:\Users\visha\Lloyds_Github\data\processed\nb10_gazette_company_features_thru_2026-07.csv
out    : C:\Users\visha\Lloyds_Github\data\processed\dashboard_bulk_gazette_2026-07.parquet


## Step 1: load the spine

Read as-is. The parquet was written by DuckDB with proper types, so unlike the CSV side of this
project there are no leading-space column names and no company numbers silently turned into
integers. The checks below confirm that rather than assume it.

In [2]:
spine = pd.read_parquet(SPINE)
print(f"rows {len(spine):,}  columns {spine.shape[1]}")

assert len(spine) == EXPECT_SPINE_ROWS, f"spine row count changed: {len(spine):,}"
assert spine["CompanyNumber"].is_unique, "spine CompanyNumber is not unique"
assert spine["CompanyNumber"].notna().all(), "spine has null CompanyNumber"
assert (spine["CompanyNumber"].str.len() == 8).all(), "spine has non 8-char company numbers"
assert (spine["CompanyNumber"] == spine["CompanyNumber"].str.upper()).all(), "spine key not uppercase"
assert spine["base_month"].nunique() == 1, "spine is not a single month snapshot"

print("key is unique, non-null, 8-char, uppercase")
print("base_month:", spine["base_month"].unique()[0], " (single month bucket, as expected)")
spine.head(3)

rows 1,531,094  columns 69


key is unique, non-null, 8-char, uppercase
base_month: 2026-07-01  (single month bucket, as expected)


,CompanyNumber,base_month,source_date,Mortgages.NumMortCharges,Mortgages.NumMortOutstanding,Mortgages.NumMortSatisfied,CompanyStatus,is_active,sector,segment,size_tier,tier_rank,SICCode.SicText_1,CompanyName,RegAddress.PostCode,company_age_years,debt_ratio,accounts_overdue,d_charges_3m,d_charges_6m,d_charges_12m,d_outstanding_3m,d_outstanding_6m,d_outstanding_12m,d_satisfied_12m,debt_ratio_trend_12m,new_charge_events_12m,months_since_last_new_charge,status_changed,months_in_current_status,...,months_since_last_confstmt,sic_changed_12m,name_changed_12m,postcode_changed_12m,contracts_won_12m,total_value_won_12m,awards_with_value_12m,months_since_last_award,d_contracts_12m,ever_won_contract,first_award_in_12m,contracts_asof_month,contracts_stale,n_charges_outstanding,n_lbg_charges_outstanding,is_lbg_client,lbg_share_of_outstanding,n_distinct_lenders,n_competitor_lenders,months_since_last_lbg_charge_created,months_since_last_lbg_satisfaction,competitor_entered_12m,lbg_charge_satisfied_6m,competitor_charge_created_6m,ever_lbg_client,primary_lender_group,score_lending,score_insolvency,score_voluntary_exit,score_growth
0,10421365,2026-07-01,2026-07-01,1,1,0,Active,True,"Technology, legal & professional",Dormant,None,NaN,74990 - Non-trading company,KNIGHT DRAGON 351 LIMITED,SE10 0ER,9.7,1.0,False,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,False,33,...,9.0,False,False,False,0,0.0,0,NaN,0,False,False,2026-05-01,True,1,0,False,0.0,0,0,NaN,NaN,False,False,False,False,None,0.012188,0.001085,0.051702,0.043409
1,10423124,2026-07-01,2026-07-01,1,1,0,Active,True,"Technology, legal & professional",Micro,Micro,1.0,70229 - Management consultancy activities othe...,KINETIC CONSTRUCTION LIMITED,MK9 2FR,9.7,1.0,False,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,9.0,False,31,...,9.0,False,False,True,0,0.0,0,NaN,0,False,False,2026-05-01,True,1,0,False,0.0,0,0,NaN,NaN,False,False,False,False,None,0.020688,0.014696,0.212209,0.110405
2,10425543,2026-07-01,2026-07-01,1,1,0,Active,True,Fast growth & emerging,Small,Small,2.0,62012 - Business and domestic software develop...,AESTHETIC NURSE SOFTWARE LIMITED,B46 1JA,9.7,1.0,False,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,2.0,False,33,...,3.0,False,False,False,0,0.0,0,NaN,0,False,False,2026-05-01,True,1,0,False,0.0,0,0,NaN,NaN,False,False,False,False,None,0.034147,0.009468,0.001063,0.001723


## Step 2: load the Gazette features

Company numbers are read as string so nothing loses a leading zero, and the two date columns are
parsed properly rather than left as text.

Four columns are dropped before the merge. Three carry no information at all once the file has been
filtered to number-matched rows (`gaz_match_method` is always `number`, `gaz_has_company_number`
always 1, `gaz_name_only_match_flag` always 0). The fourth, `company_name`, duplicates the spine's
`CompanyName`, and keeping a second name column is exactly how a name-based join creeps in later.

In [3]:
gaz = pd.read_csv(GAZ, dtype={"CompanyNumber": str},
                  parse_dates=["gaz_first_notice_date", "gaz_latest_notice_date"])
print(f"rows {len(gaz):,}  columns {gaz.shape[1]}")

assert len(gaz) == EXPECT_GAZ_ROWS, f"gazette row count changed: {len(gaz):,}"
assert gaz["CompanyNumber"].is_unique, "gazette CompanyNumber is not unique"
assert gaz["CompanyNumber"].notna().all(), "gazette has null CompanyNumber"

# These are constant post-filter, so they are noise rather than signal. Verified, not assumed.
for col, const in [("gaz_match_method", "number"), ("gaz_has_company_number", 1),
                   ("gaz_name_only_match_flag", 0)]:
    vals = gaz[col].unique()
    print(f"  {col:26s} -> {vals}  (constant: {len(vals) == 1 and vals[0] == const})")

DROP_COLS = ["company_name", "gaz_match_method", "gaz_has_company_number", "gaz_name_only_match_flag"]
gaz = gaz.drop(columns=DROP_COLS)
print(f"\ndropped {len(DROP_COLS)} columns -> {gaz.shape[1]} remaining "
      f"({gaz.shape[1] - 1} gaz features plus the key)")

rows 109,333  columns 56
  gaz_match_method           -> ['number']  (constant: True)
  gaz_has_company_number     -> [1]  (constant: True)
  gaz_name_only_match_flag   -> [0]  (constant: True)

dropped 4 columns -> 52 remaining (51 gaz features plus the key)


## Step 3: pre-merge validation

The join key needs no cleaning. Both sides are already 8-character, uppercase and zero-padded, and
during inspection a `strip + upper + zfill(8)` pass rescued exactly zero additional matches. Running
one anyway would risk corrupting the `SC`, `NI` and `R` prefixed numbers, so it is deliberately
omitted and the assumption is tested instead.

In [4]:
overlap = (set(spine.columns) & set(gaz.columns)) - {"CompanyNumber"}
assert not overlap, f"unexpected column collision, would create _x/_y suffixes: {sorted(overlap)}"
print("column-name collisions other than the key:", sorted(overlap) or "none")

fmt = gaz["CompanyNumber"]
assert (fmt.str.len() == 8).all() and (fmt == fmt.str.upper()).all(), "gazette key format is off"
print("gazette key: all 8-char, uppercase")

spine_keys = set(spine["CompanyNumber"])
in_universe = gaz["CompanyNumber"].isin(spine_keys)
print(f"\ngazette rows that will match : {in_universe.sum():,} / {len(gaz):,} "
      f"({100 * in_universe.mean():.1f}%)")
print(f"gazette rows out of universe : {(~in_universe).sum():,}")

# Confirm no match is being lost to formatting, as opposed to genuinely being out of universe.
rescued = gaz.loc[~in_universe, "CompanyNumber"].str.strip().str.upper().str.zfill(8).isin(spine_keys).sum()
assert rescued == 0, f"{rescued} rows would be rescued by normalising the key, investigate first"
print("rows rescuable by key normalisation:", rescued, "(so the unmatched are genuinely out of universe)")

# Some gazette columns are already null for reasons that have nothing to do with the merge:
# a notice with no parseable postcode, or a company with no petition preceding an order.
# Capture that count now, on the rows that will actually match, so the post-merge check can
# prove the merge preserved them exactly rather than compare against a guessed constant.
UPSTREAM_NULLS = gaz.loc[in_universe].isna().sum()
print("\ngenuine nulls already present on the matching rows:")
print(UPSTREAM_NULLS[UPSTREAM_NULLS > 0].to_string())

column-name collisions other than the key: none
gazette key: all 8-char, uppercase



gazette rows that will match : 19,525 / 109,333 (17.9%)
gazette rows out of universe : 89,808


rows rescuable by key normalisation: 0 (so the unmatched are genuinely out of universe)

genuine nulls already present on the matching rows:
gaz_notice_postcode            9592
gaz_notice_postcode_area       9592
gaz_days_petition_to_order    17420


### The unmatched Gazette records

89,808 Gazette companies have no home in this universe. That is expected rather than a fault: the
spine is a filtered universe of three sector groups, and roughly 57% of the unmatched sit in the full
Companies House bulk but outside that filter (mostly already in liquidation), while the rest have
been dissolved and removed from the bulk entirely. LLP and other prefixed types (`OC`, `CE`, `SO`,
`RS` and so on) are out of universe by construction.

They are dropped by the left join, but written to a side file first so the loss is auditable in the
write-up rather than invisible.

In [5]:
unmatched = gaz.loc[~in_universe, ["CompanyNumber", "gaz_severity_tier",
                                   "gaz_latest_notice_date", "gaz_latest_notice_url"]]
unmatched.to_csv(UNMATCHED_OUT, index=False)
print(f"wrote {len(unmatched):,} unmatched gazette records -> {UNMATCHED_OUT.name}")
print("\nby severity:")
print(unmatched["gaz_severity_tier"].value_counts().to_string())
print("\nby company-number prefix:")
print(unmatched["CompanyNumber"].str.extract(r"^([A-Z]{1,2})", expand=False)
      .fillna("(numeric)").value_counts().head(8).to_string())

wrote 89,808 unmatched gazette records -> gaz_unmatched_2026-07.csv

by severity:
gaz_severity_tier
formal_insolvency    79444
terminal              8315
early_warning         1893
none                   156

by company-number prefix:
CompanyNumber
(numeric)    83368
SC            4496
NI            1195
OC             599
CE              40
SO              25
RS              21
OE              17


## Step 4: the left join

Left, because the spine defines the universe the dashboard renders and every one of its rows must
survive. An inner join would collapse it to 19,525 distressed companies and destroy the dashboard.
An outer join would inject roughly 90k companies with nulls across all 69 spine columns, including
`sector`, `segment` and the scores.

`validate="one_to_one"` is the important argument: both sides are unique on the key today, and if
that ever stops being true the merge should fail rather than quietly multiply rows.

In [6]:
merged = spine.merge(gaz, on="CompanyNumber", how="left",
                     validate="one_to_one", indicator="_merge_src")

merged["gaz_matched"] = (merged["_merge_src"] == "both").astype("int8")
matched_n = int(merged["gaz_matched"].sum())
merged = merged.drop(columns="_merge_src")

print(f"rows {len(merged):,}  columns {merged.shape[1]}")
print(f"matched   : {matched_n:,} ({100 * matched_n / len(merged):.2f}% of the universe)")
print(f"unmatched : {len(merged) - matched_n:,}")

assert len(merged) == EXPECT_SPINE_ROWS, "row count changed during the merge"
assert merged["CompanyNumber"].is_unique, "merge duplicated the key"
assert matched_n == EXPECT_MATCHED, f"match count changed: {matched_n:,} vs {EXPECT_MATCHED:,}"

rows 1,531,094  columns 121
matched   : 19,525 (1.28% of the universe)
unmatched : 1,511,569


## Step 5: fill rules, by column family

A blanket `fillna(0)` would be wrong here and quietly damaging. For a company with no notice,
`gaz_days_since_latest_notice = 0` reads as "had a notice today", which inverts the signal it is
meant to carry. So the columns are split into families and filled accordingly:

| Family | Fill | Reasoning |
|---|---|---|
| counts and binary flags | `0` | no notice means zero notices, a real value |
| `gaz_max_distress_stage` | `0` | 0 already means "none" in the existing 0 to 5 scale |
| `gaz_severity_tier` | `"none"` | already a valid level in the data |
| `gaz_current_stage` | `"no_notice"` | its vocabulary has no "none" level, and "other" means something specific |
| dates | left null | there is no first or latest notice date |
| day counts and spans | left null | a sentinel here would read as recent distress |
| postcode, URLs | left null | nothing to record |

Every column is assigned to exactly one family and the assignment is printed, so the rule applied to
each column is visible rather than buried in the code.

In [7]:
DATE_COLS = ["gaz_first_notice_date", "gaz_latest_notice_date"]
NULL_NUMERIC = ["gaz_days_since_latest_notice", "gaz_days_since_first_notice",
                "gaz_notice_span_days", "gaz_days_petition_to_order"]
NULL_TEXT = ["gaz_notice_postcode", "gaz_notice_postcode_area",
             "gaz_latest_notice_url", "gaz_source_notice_urls"]
CATEGORICAL_FILL = {"gaz_severity_tier": "none", "gaz_current_stage": "no_notice"}

gaz_cols = [c for c in merged.columns if c.startswith("gaz_") and c != "gaz_matched"]
ZERO_FILL = [c for c in gaz_cols
             if c not in DATE_COLS + NULL_NUMERIC + NULL_TEXT + list(CATEGORICAL_FILL)]

# Every gaz column must land in exactly one family, or the fill is incomplete.
assigned = set(DATE_COLS) | set(NULL_NUMERIC) | set(NULL_TEXT) | set(CATEGORICAL_FILL) | set(ZERO_FILL)
assert assigned == set(gaz_cols), f"unassigned columns: {sorted(set(gaz_cols) ^ assigned)}"

print(f"zero-filled ({len(ZERO_FILL)}):")
for c in ZERO_FILL:
    print("   ", c)
print(f"\ncategorical fill ({len(CATEGORICAL_FILL)}): {CATEGORICAL_FILL}")
print(f"left null ({len(DATE_COLS + NULL_NUMERIC + NULL_TEXT)}): "
      f"{DATE_COLS + NULL_NUMERIC + NULL_TEXT}")

zero-filled (39):
    gaz_notice_count_total
    gaz_distinct_notice_type_count
    gaz_winding_up_petition_count
    gaz_winding_up_order_count
    gaz_voluntary_liquidation_count
    gaz_liquidation_notice_count
    gaz_administration_count
    gaz_creditors_process_count
    gaz_dividend_notice_count
    gaz_closing_notice_count
    gaz_notice_count_12m
    gaz_notice_count_24m
    gaz_max_distress_stage
    gaz_has_prohibited_name_reuse
    gaz_has_petition_dismissed
    gaz_has_cross_border_insolvency
    gaz_has_other_insolvency
    gaz_receiver_appointed_flag
    gaz_partnership_insolvency_flag
    gaz_postcode_present_flag
    gaz_has_any_notice
    gaz_has_winding_up_petition
    gaz_has_winding_up_order
    gaz_has_liquidation
    gaz_has_administration
    gaz_has_creditors_process
    gaz_has_dividend_notice
    gaz_has_closing_notice
    gaz_compulsory_liquidation_flag
    gaz_voluntary_liquidation_flag
    gaz_court_involved_flag
    gaz_recent_notice_90d_flag
    gaz_rec

In [8]:
for col, value in CATEGORICAL_FILL.items():
    merged[col] = merged[col].fillna(value)

for col in ZERO_FILL:
    filled = merged[col].fillna(0)
    # These are counts, flags and one 0-5 ordinal, so an integer type is the honest one.
    merged[col] = filled.astype("int32") if filled.max() > 1 else filled.astype("int8")

print("counts and flags filled with 0 and cast to integer")
print("categoricals filled\n")
print(merged["gaz_severity_tier"].value_counts().to_string())
print()
print(merged["gaz_current_stage"].value_counts().to_string())

counts and flags filled with 0 and cast to integer
categoricals filled

gaz_severity_tier
none                 1511596
formal_insolvency      17136
terminal                1873
early_warning            489

gaz_current_stage
no_notice            1511569
formal_insolvency      18698
petition                 473
creditor_process         189
other                    113
closing                   27
dividend                  25


## Step 6: the derived columns

Four additions. `gaz_asof_date` stamps the Gazette clock into the data so the 30-day gap between the
two anchors is visible rather than notebook folklore, which matters because the spine's own day
counts (`days_to_next_accounts_due`) are measured from a different point and the two should never be
combined in one expression.

`gaz_new_in_july_flag` marks the companies whose first ever notice landed in July. These are not a
problem to exclude, they are the most valuable rows in the file: for most of them Companies House
still says "Active" because its status field lags publication, so the Gazette is carrying information
the spine cannot produce on its own.

In [9]:
merged["gaz_asof_date"] = GAZ_ASOF

first, last = merged["gaz_first_notice_date"], merged["gaz_latest_notice_date"]
merged["gaz_new_in_july_flag"] = (first.between(JULY_START, JULY_END)).astype("int8")
merged["gaz_july_notice_flag"] = (last.between(JULY_START, JULY_END)).astype("int8")

print(f"first-ever notice in July : {int(merged['gaz_new_in_july_flag'].sum()):,}")
print(f"latest notice in July     : {int(merged['gaz_july_notice_flag'].sum()):,}")

nj = merged[merged["gaz_new_in_july_flag"] == 1]
print("\nCompanies House status of the first-in-July companies:")
print(nj["CompanyStatus"].value_counts().to_string())
print(f"\nof those, never previously flagged distressed by CH: "
      f"{int((~nj['ever_distressed_before']).sum()):,}")

first-ever notice in July : 740
latest notice in July     : 868



Companies House status of the first-in-July companies:
CompanyStatus
Active                             656
Active - Proposal to Strike off     69
Liquidation                         10
Voluntary Arrangement                3
In Administration                    2

of those, never previously flagged distressed by CH: 586


## Step 7: post-merge validation

The checks that matter, run against the finished frame. The row-count and uniqueness checks catch key
duplication, the null-parity check catches a merge that silently damaged the spine, and the last one
is the leakage guard: it fails if a Gazette file covering a later period is ever joined by mistake.

In [10]:
checks = []

checks.append(("row count preserved", len(merged) == EXPECT_SPINE_ROWS, f"{len(merged):,}"))
checks.append(("key still unique", merged["CompanyNumber"].is_unique, ""))
checks.append(("match count as expected", int(merged["gaz_matched"].sum()) == EXPECT_MATCHED,
               f"{int(merged['gaz_matched'].sum()):,}"))

# No spine column may have gained a null.
spine_nulls_before = spine.isna().sum()
spine_nulls_after = merged[spine.columns].isna().sum()
same = bool((spine_nulls_before == spine_nulls_after).all())
checks.append(("no spine column gained nulls", same,
               "" if same else str((spine_nulls_after - spine_nulls_before)
                                   .pipe(lambda s: s[s != 0]).to_dict())))

zero_null = int(merged[ZERO_FILL].isna().sum().sum())
checks.append(("counts and flags have no nulls", zero_null == 0, str(zero_null)))

# Dates must be null exactly where there is no match.
unmatched_mask = merged["gaz_matched"] == 0
for col in DATE_COLS:
    ok = bool((merged[col].isna() == unmatched_mask).all())
    checks.append((f"{col} null exactly on unmatched", ok, ""))

# Day counts and text: null on every unmatched row, plus exactly the genuine nulls the source
# already carried on the matching rows. Nothing invented, nothing silently filled.
for col in NULL_NUMERIC + NULL_TEXT:
    genuine = int(UPSTREAM_NULLS[col])
    extra = int(merged[col].isna().sum() - unmatched_mask.sum())
    checks.append((f"{col} nulls reconcile", extra == genuine, f"{extra:,} upstream, expected {genuine:,}"))

# The leakage guard.
latest = merged["gaz_latest_notice_date"].max()
checks.append(("no notice later than the gazette anchor", latest <= GAZ_ASOF, str(latest.date())))

for name, ok, detail in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name:46s} {detail}")

failed = [n for n, ok, _ in checks if not ok]
assert not failed, f"validation failed: {failed}"
print(f"\nall {len(checks)} checks passed")

  [PASS] row count preserved                            1,531,094
  [PASS] key still unique                               
  [PASS] match count as expected                        19,525
  [PASS] no spine column gained nulls                   
  [PASS] counts and flags have no nulls                 0
  [PASS] gaz_first_notice_date null exactly on unmatched 
  [PASS] gaz_latest_notice_date null exactly on unmatched 
  [PASS] gaz_days_since_latest_notice nulls reconcile   0 upstream, expected 0
  [PASS] gaz_days_since_first_notice nulls reconcile    0 upstream, expected 0
  [PASS] gaz_notice_span_days nulls reconcile           0 upstream, expected 0
  [PASS] gaz_days_petition_to_order nulls reconcile     17,420 upstream, expected 17,420
  [PASS] gaz_notice_postcode nulls reconcile            9,592 upstream, expected 9,592
  [PASS] gaz_notice_postcode_area nulls reconcile       9,592 upstream, expected 9,592
  [PASS] gaz_latest_notice_url nulls reconcile          0 upstream, expected 0
  [

### Spot check

One company carried end to end, verified against its live notice during the inspection pass, and one
company with no notice at all to confirm the fill rules read correctly.

In [11]:
row = merged.loc[merged["CompanyNumber"] == "04414254"].iloc[0]
for c in ["CompanyNumber", "CompanyName", "CompanyStatus", "segment", "gaz_matched",
          "gaz_notice_count_total", "gaz_severity_tier", "gaz_current_stage",
          "gaz_first_notice_date", "gaz_latest_notice_date", "gaz_days_since_latest_notice",
          "gaz_new_in_july_flag", "gaz_latest_notice_url"]:
    print(f"  {c:30s} {row[c]}")

none_row = merged.loc[merged["gaz_matched"] == 0].iloc[0]
print("\nan unmatched company, to confirm the fill rules:")
for c in ["CompanyNumber", "CompanyName", "gaz_matched", "gaz_notice_count_total",
          "gaz_has_any_notice", "gaz_max_distress_stage", "gaz_severity_tier",
          "gaz_current_stage", "gaz_latest_notice_date", "gaz_days_since_latest_notice"]:
    print(f"  {c:30s} {none_row[c]}")

  CompanyNumber                  04414254
  CompanyName                    MEC (GB) LTD.
  CompanyStatus                  Active
  segment                        Micro
  gaz_matched                    1
  gaz_notice_count_total         3
  gaz_severity_tier              terminal
  gaz_current_stage              formal_insolvency
  gaz_first_notice_date          2026-06-30 00:00:00
  gaz_latest_notice_date         2026-07-08 00:00:00
  gaz_days_since_latest_notice   23.0
  gaz_new_in_july_flag           0
  gaz_latest_notice_url          https://www.thegazette.co.uk/id/notice/5170567



an unmatched company, to confirm the fill rules:
  CompanyNumber                  10421365
  CompanyName                    KNIGHT DRAGON 351 LIMITED
  gaz_matched                    0
  gaz_notice_count_total         0
  gaz_has_any_notice             0
  gaz_max_distress_stage         0
  gaz_severity_tier              none
  gaz_current_stage              no_notice
  gaz_latest_notice_date         NaT
  gaz_days_since_latest_notice   nan


## Step 8: export

Parquet, to match the spine. It keeps the integer and date types that were just set, which a CSV
would throw away, and the file lands at a fraction of the CSV size.

In [12]:
merged.to_parquet(OUT, index=False, compression="snappy")
size_mb = OUT.stat().st_size / 1e6
print(f"wrote {OUT}")
print(f"  {len(merged):,} rows x {merged.shape[1]} columns, {size_mb:.1f} MB")

check = pd.read_parquet(OUT, columns=["CompanyNumber", "gaz_matched", "gaz_severity_tier"])
assert len(check) == len(merged) and check["CompanyNumber"].is_unique
print(f"  read back cleanly: {len(check):,} rows, key unique")

wrote C:\Users\visha\Lloyds_Github\data\processed\dashboard_bulk_gazette_2026-07.parquet
  1,531,094 rows x 124 columns, 92.5 MB


  read back cleanly: 1,531,094 rows, key unique


## Step 9: what the unified file now contains

The headline is not the 1.28% match rate. It is where those matches sit. Most of them restate what
Companies House already says, because a company in liquidation with a formal insolvency notice is not
news. The value is concentrated in the companies the spine still calls Active.

In [13]:
print(f"universe             : {len(merged):,} companies")
print(f"with a Gazette record: {int(merged['gaz_matched'].sum()):,} "
      f"({100 * merged['gaz_matched'].mean():.2f}%)")
print(f"columns              : {merged.shape[1]} "
      f"({spine.shape[1]} spine + {merged.shape[1] - spine.shape[1]} gazette)")

active = merged[merged["is_active"] & (merged["gaz_matched"] == 1)]
print(f"\nstill Active in Companies House but carrying a Gazette notice: {len(active):,}")
print(pd.crosstab(active["gaz_severity_tier"], active["gaz_new_in_july_flag"],
                  colnames=["first notice in July"]).to_string())

print("\nseverity across all matched companies:")
print(merged.loc[merged["gaz_matched"] == 1, "gaz_severity_tier"].value_counts().to_string())

print("\nmatched companies by segment:")
print(pd.crosstab(merged["segment"], merged["gaz_matched"]).to_string())

universe             : 1,531,094 companies
with a Gazette record: 19,525 (1.28%)
columns              : 124 (69 spine + 55 gazette)

still Active in Companies House but carrying a Gazette notice: 1,232
first notice in July    0    1
gaz_severity_tier             
early_warning         267   63
formal_insolvency     211  525
none                   21    0
terminal               77   68

severity across all matched companies:
gaz_severity_tier
formal_insolvency    17136
terminal              1873
early_warning          489
none                    27

matched companies by segment:


gaz_matched       0      1
segment                   
Dormant      180716    470
Large         29966    760
Medium         2004     22
Micro        555858   7519
No Filings   337540    507
Small        396136  10119
Subsidiary     9320    128
Unknown          25      0


### Known limitations, recorded rather than buried

- **Coverage is not proof of health.** 98.7% of companies have no Gazette record. That is the
  expected state, not missing data, and the zero-filled counts should be read that way.
- **Extraction was sampled, not exhaustively audited.** 30 of the 656 Active-with-a-July-notice
  companies were checked against their live notice pages, comparing the pipeline's regex-extracted
  number against the Gazette's own tagged `gazorg:companyNumber` and `gazorg:name` fields. All 30
  matched on both. Zero errors in 30 puts the true error rate plausibly under about 10%, not at zero.
  The check tests false positives only, it does not detect notices that were missed.
- **Out-of-universe loss is real.** 89,808 Gazette companies are dropped, listed in
  `gaz_unmatched_2026-07.csv`. Roughly 57% sit in the full Companies House bulk but outside this
  sector filter, the rest are dissolved and gone from the bulk. LLPs and other prefixed types cannot
  match by construction.
- **Two clocks.** Spine day counts and `gaz_days_since_*` are measured from different anchors, 30
  days apart, both stamped in the file. Do not combine them in one expression.
- **Contemporaneous, not predictive.** A July notice and July spine features describe the same month.
  Fine for predicting a later month, circular if used to "predict" the same month.

### The notices with no company number, and what that costs

The largest known gap in this pipeline sits upstream of the join, in the notice file these features
were aggregated from. Of the 291,045 Gazette notices crawled, **129,918 (44.6%) carry no company
number**. Every one of them is dropped before a company is ever identified, and the reason is
uniform: the notice text simply does not print a registered number. None were lost to a missing
date. Of those, 9,120 are not company notices at all (honours lists, Crown Office and Deputy
Lieutenant commissions, which reveal that the crawl's category filter is wider than corporate
insolvency and could usefully be tightened). The remaining **120,798 are genuine company insolvency
notices**.

The loss is not evenly spread, which is what makes it matter. It falls overwhelmingly on particular
notice types: **72% of all "Resolutions for Winding-up" and 98% of "Notices to Creditors" have no
number**, against under 1% of "Appointment of Liquidators". So the gap is systematic rather than
random, and it bites hardest on the early and procedural steps of an insolvency rather than the
final ones. The saving grace is that 88.4% of these notices belong to companies that are already in
the dataset through some *other* notice that did carry a number. The practical consequence is
therefore incomplete histories rather than invisible companies: **18,225 dropped notices belong to
the 19,525 companies in this file, affecting 12,905 of them (66%)**, and of the 1,232 companies that
are still Active while carrying a notice, **556 (45%) are missing at least one**, mostly a
Resolution for Winding-up. Only around 741 companies could be entirely absent because of this.

**How to read the affected columns.** `gaz_notice_count_total`, `gaz_notice_count_12m` and
`gaz_distinct_notice_type_count` are **floors, not totals**, and a company's rendered timeline may
be missing steps. `gaz_max_distress_stage` and `gaz_current_stage` may also sit below a company's
true position where the missing notice was the furthest-advanced one. None of this is a regression
introduced by this notebook: the feature file has always been 100% number-matched
(`gaz_match_method` is "number" on every row), so these notices were never present in any earlier
build either.

**The fix, if it is ever worth the cost.** Each notice's own page exposes the number as a structured
tagged field (`gazorg:companyNumber`), available as XHTML, RDF or JSON-LD, and this is authoritative
rather than inferred. The search feed does not carry it, so recovery means one request per notice.
Fetching all 120,798 is roughly 34 hours at a polite one request per second. A targeted pass is far
cheaper: use name similarity purely to *choose which URLs to open*, never to assign a company, then
take the number from the fetched page and discard anything outside the universe. That recovers the
~18,000 notices belonging to companies already here in about five hours, and the 740 missing from
the still-Active companies in about thirteen minutes. Because the number always comes from the
page rather than from the name, this introduces none of the false-positive risk that name matching
would.